In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))
import logging
import warnings
import numpy as np
import re
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from src.build_dataset import get_file_pairs, merge_qa_data, detect_exercise_type, find_answer_index, apply_reference_tag

# --- Setup Warnings ---
warnings.filterwarnings('ignore', category=SyntaxWarning, message='invalid escape sequence')

# --- Setup Logger ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# --- Load Environment Variables ---
load_dotenv()

True

In [2]:
data_path = os.getenv("DATA_DIR")

if data_path:
    data_dir = Path(data_path)
    logger.info(f"Ξεκινάει η αναζήτηση στον φάκελο: {data_dir}")
    
    all_pairs = get_file_pairs (data_dir, target_school="GEL")
    logger.info(f"Βρέθηκαν συνολικά {len(all_pairs)} ζευγάρια αρχείων (JSON/MD).")
else:
    logger.error("Το DATA_DIR δεν βρέθηκε στο .env αρχείο!")

2026-04-06 16:02:21 - INFO - Ξεκινάει η αναζήτηση στον φάκελο: /home/eleni/panellinies/panellinies_exams_dataset/data
2026-04-06 16:02:21 - INFO - Βρέθηκαν συνολικά 59 ζευγάρια αρχείων (JSON/MD).


In [3]:
main_dataset = []

for pair in all_pairs:
    json_path = pair["json"]
    md_path = pair["md"]
    
    qa_list = merge_qa_data(json_path, md_path)
    
    main_dataset.extend(qa_list)

logger.info (f"Η ενοποίηση ολοκληρώθηκε! Βρέθηκαν συνολικά {len(main_dataset)} ερωτήσεις-απαντήσεις!")

2026-04-06 16:02:25 - INFO - Η ενοποίηση ολοκληρώθηκε! Βρέθηκαν συνολικά 1318 ερωτήσεις-απαντήσεις!


In [4]:
subject_translation = {
    "nea_ellinika": "greek_language",
    "arxaia": "ancient_greek",
    "istoria": "history",
    "latinika": "latin",
    "biologia": "biology",
    "fysiki": "physics",
    "ximeia": "chemistry",
    "pliroforiki": "computer_science",
    "arxes_oikonomikis_theorias": "economics",
    "mathimatika": "mathematics"
}

In [5]:
for item in main_dataset:
    q_text = item.get("question","")
    q_choices = item.get("choices",[])
    ans_text = item.get("answer","")
    images_list = item.get("images", [])
    marks = item.get("mark", [])
    
    form_type = detect_exercise_type(q_text,q_choices)
    item["format"] = form_type
    ans_idx = find_answer_index(q_choices,ans_text)
    item["answer_index"] = ans_idx
    item["reference"] = apply_reference_tag(item)
    
    old_subj = item.get("subject", "")
    new_subj = subject_translation.get(old_subj, old_subj)
    item["subject"] = new_subj
    
    #parsing image description and transcription
    all_descriptions = []
    all_transcriptions = []
    all_paths = []
    
    for img_dict in images_list:
        desc = img_dict.get("description","")
        if desc:
            all_descriptions.append(desc)
        transc = img_dict.get("transcription",[])
        if transc and isinstance(transc, list):
            joined_transc = ", ".join(transc)
            all_transcriptions.append(joined_transc)
        
        img_path = img_dict.get("path", "")
        if img_path:
            all_paths.append(img_path)
    
    mark_list = []
    
    for mark_text in marks:
        match = re.search(r'\d+\.?\d*', str(mark_text))
        if match:
            num_str = match.group()
            if "." in num_str:
                mark_list.append(float(num_str))
            else:
                mark_list.append(int(num_str))
    
    if len(mark_list) == 1:
        item["points"] = mark_list[0]
    elif len(mark_list) > 1:
        item["points"] = sum(mark_list)
    else:
        item["points"] = None
            
    item["image_description"] = " | ".join(all_descriptions)
    item["image_transcription"] = " | ".join(all_transcriptions)
    item["images"] = all_paths
    item.pop("mark", None)
    
    year = item.get("year", "")
    old_id = item.get("id", "")
    school_type = str(item.get("school_type", "gel")).lower()
    item["id"] = f"{new_subj}_{school_type}_{year}_{old_id}"

In [6]:
images_found = 0
print("--- Ερωτήσεις που βρέθηκαν να έχουν εικόνες ---")

for item in main_dataset:
    imgs = item.get("images", [])
    
    if isinstance(imgs, list) and len(imgs) > 0:
        images_found += 1
        print(f"ID: {item.get('id')} στο μάθημα {item.get('subject')} ({item.get('year')}) - Περιέχει {len(imgs)} εικόνα/ες")

print(f"\nΣυνολικά βρέθηκαν {images_found} ερωτήσεις (IDs) με εικόνες.")

--- Ερωτήσεις που βρέθηκαν να έχουν εικόνες ---
ID: physics_gel_2020_A3 στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_B1.α στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_B1.β στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_B3.α στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_B3.β στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_Γ1 στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_Γ2 στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_Γ3 στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_Γ4 στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_Δ1 στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_Δ2 στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_Δ3 στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_Δ4 στο μάθημα physics (2020) - Περιέχει 1 εικόν

In [7]:
results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

output_file = results_dir / "panellinies_dataset.xlsx"

In [8]:
df = pd.DataFrame(main_dataset)
df = df.rename (columns={"answer": "answer_text"})
my_columns = [
    "id",
    "subject",
    "format",
    "reference",
    "question",
    "input",
    "images",
    "choices",
    "answer_text",
    "answer_index",
    "image_description",
    "image_transcription",
    "points",
    "year",
    "school_type"
]

df = df[my_columns]

In [9]:
df.to_excel(output_file, index=False)

logger.info (f"Tο αρχείο δημιουργήθηκε επιτυχώς στο: {output_file.resolve()}!")

df.head()

2026-04-06 16:03:08 - INFO - Tο αρχείο δημιουργήθηκε επιτυχώς στο: /home/eleni/panellinies/panellinies_exams_dataset/results/panellinies_dataset.xlsx!


,id,subject,format,reference,question,input,images,choices,answer_text,answer_index,image_description,image_transcription,points,year,school_type
0,physics_gel_2020_A1,physics,multiple_choice,none,Στις ερωτήσεις Α1-Α4 να γράψετε στο τετράδιό σ...,,[],"[α. $V$, β. $2V$, γ. $\frac{V}{2}$, δ. $\frac{...",γ,2.0,,,5.0,2020,GEL
1,physics_gel_2020_A2,physics,multiple_choice,none,Στις ερωτήσεις Α1-Α4 να γράψετε στο τετράδιό σ...,,[],"[α. $B$, β. $2B$, γ. $\frac{B}{2}$, δ. $\frac{...",α,0.0,,,5.0,2020,GEL
2,physics_gel_2020_A3,physics,multiple_choice,multimodal,Στις ερωτήσεις Α1-Α4 να γράψετε στο τετράδιό σ...,,[images/them_fysiki_gel_2020/img1/them_fysiki_...,"[α. $P_1 < P_2$, β. $P_1 = P_2$, γ. $P_1 > P_2...",γ,2.0,A diagram showing a section of a pipe lying on...,Σχήμα 1,5.0,2020,GEL
3,physics_gel_2020_A4,physics,multiple_choice,none,Στις ερωτήσεις Α1-Α4 να γράψετε στο τετράδιό σ...,,[],[α. το πλάτος της σύνθετης ταλάντωσης είναι αρ...,δ,3.0,,,5.0,2020,GEL
4,physics_gel_2020_A5.α,physics,true_false,none,"Να χαρακτηρίσετε τις προτάσεις που ακολουθούν,...",,[],"[α. Σωστό, β. Λάθος]",Σωστό,0.0,,,1.0,2020,GEL


In [11]:
def normalize_for_compare(value):
    if value is None:
        return ""

    if isinstance(value, (list, tuple, np.ndarray)):
        return str([str(x).strip() for x in list(value)])

    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    return str(value).strip()

In [12]:
def compare(current_df, reference_file):
    """Συγκρίνει το current dataset με ένα reference Excel αρχείο."""

    if current_df.empty:
        logger.error("❌ Δεν μπορεί να γίνει σύγκριση: το current dataset είναι κενό.")
        return None

    reference_file = Path(reference_file)

    if not reference_file.exists():
        logger.error(f"❌ Το reference αρχείο δε βρέθηκε: {reference_file}")
        return None

    logger.info(f"🔍 Σύγκριση με το αρχείο: {reference_file}")

    ref_df = pd.read_excel(reference_file)

    if "id" not in current_df.columns:
        logger.error("❌ Η στήλη 'id' λείπει από το current dataset.")
        return None

    if "id" not in ref_df.columns:
        logger.error("❌ Η στήλη 'id' λείπει από το reference dataset.")
        return None

    merged = pd.merge(
        current_df,
        ref_df,
        on="id",
        suffixes=("_cur", "_ref"),
        how="outer",
        indicator=True
    )

    only_cur = merged[merged["_merge"] == "left_only"]
    only_ref = merged[merged["_merge"] == "right_only"]
    both = merged[merged["_merge"] == "both"]

    print("📊 Statistics:")
    print(f"   - Match: {len(both)}")
    print(f"   - Only in Current: {len(only_cur)}")
    print(f"   - Only in Reference: {len(only_ref)}")

    cols_to_check = [
        "subject",
        "format",
        "reference",
        "question",
        "input",
        "choices",
        "answer_text",
        "answer_index",
        "image_description",
        "image_transcription",
        "points",
        "year",
        "school_type"
    ]

    for col in cols_to_check:
        col_cur = f"{col}_cur"
        col_ref = f"{col}_ref"

        if col_cur not in merged.columns or col_ref not in merged.columns:
            print(f"⚠️ Η στήλη '{col}' δεν υπάρχει και στα δύο datasets. Παραλείπεται.")
            continue

        cur_series = both[col_cur].apply(normalize_for_compare)
        ref_series = both[col_ref].apply(normalize_for_compare)

        mismatch = cur_series != ref_series

        if mismatch.any():
            print(f"❌ Mismatch in column '{col}': {mismatch.sum()} differences.")
            sample_diffs = both.loc[mismatch, ["id", col_cur, col_ref]].head(5)

            for _, row in sample_diffs.iterrows():
                print(f"   ID: {row['id']}")
                print(f"      current  = {row[col_cur]}")
                print(f"      reference= {row[col_ref]}")
        else:
            print(f"✅ Column '{col}' matches perfectly.")

    return {
        "merged": merged,
        "only_current": only_cur,
        "only_reference": only_ref,
        "matched": both
    }

In [13]:
reference_file = Path("/home/eleni/panellinies/panellinies_exams_dataset/results/panellinies_dataset.xlsx")

compare_results = compare(df, reference_file)

2026-04-06 16:03:27 - INFO - 🔍 Σύγκριση με το αρχείο: /home/eleni/panellinies/panellinies_exams_dataset/results/panellinies_dataset.xlsx


📊 Statistics:
   - Match: 1318
   - Only in Current: 0
   - Only in Reference: 0
✅ Column 'subject' matches perfectly.
✅ Column 'format' matches perfectly.
✅ Column 'reference' matches perfectly.
❌ Mismatch in column 'question': 1 differences.
   ID: physics_gel_2024_Γ2
      current  = Εγκάρσιο αρμονικό κύμα, πλάτους $A$ και μήκους κύματος $\lambda$, διαδίδεται χωρίς απώλειες ενέργειας σε ομογενές γραμμικό ελαστικό μέσο μεγάλου μήκους που ταυτίζεται με τον οριζόντιο ημιάξονα $Ox$ προς τη θετική κατεύθυνση, όπως φαίνεται στο σχήμα.
Το κύμα παράγεται από πηγή που βρίσκεται στο σημείο $O$ στη θέση $x=0$ του ελαστικού μέσου και το οποίο αρχίζει να ταλαντώνεται με θετική ταχύτητα τη χρονική στιγμή $t=0$ σύμφωνα με την εξίσωση $y = A \cdot \eta\mu \omega t$.
Το υλικό σημείο $O$ κατά τη διάρκεια της ταλάντωσής του διέρχεται 60 φορές το λεπτό από τη θέση ισορροπίας του.
Κάποια χρονική στιγμή που το υλικό σημείο $O$ βρίσκεται στην ακραία αρνητική του απομάκρυνση ($y = -A$) από την αρχική θέση 

In [15]:
compare_results["only_current"][["id"]].head(20)

,id


In [16]:
compare_results["only_reference"][["id"]].head(20)

,id


In [17]:
compare_results["only_current"].to_excel("../results/only_current.xlsx", index=False)
compare_results["only_reference"].to_excel("../results/only_reference.xlsx", index=False)
compare_results["matched"].to_excel("../results/matched.xlsx", index=False)

print("✅ Αποθηκεύτηκαν τα compare outputs στον φάκελο results.")

✅ Αποθηκεύτηκαν τα compare outputs στον φάκελο results.


In [ ]:
####def push_to_hub(df, with_images=False):
    """Ανεβάζει το processed dataset στο Hugging Face Hub."""
    
    repo_id = os.getenv("HF_REPO_ID")
    token = os.getenv("HF_TOKEN")
    is_private = os.getenv("HF_PRIVATE_REPO", "True").lower() == "true"
    gated_setting = os.getenv("HF_GATED_REPO", "False").lower()

    if not repo_id:
        print("❌ Error: HF_REPO_ID not found in .env")
        return

    if not token:
        print("❌ Error: HF_TOKEN not found in .env")
        return

    print(f"📤 Preparing to push to Hugging Face Hub: {repo_id}")

    try:
        if not repo_exists(repo_id=repo_id, token=token, repo_type="dataset"):
            create_repo(
                repo_id=repo_id,
                token=token,
                private=is_private,
                repo_type="dataset"
            )
            print(f"✅ Created dataset repo: {repo_id}")

        if gated_setting in ["true", "manual"]:
            api = HfApi()
            gated_value = True if gated_setting == "true" else "manual"
            api.update_repo_settings(
                repo_id=repo_id,
                gated=gated_value,
                token=token,
                repo_type="dataset"
            )
            print(f"🔒 Updated gated setting: {gated_value}")

    except Exception as e:
        print(f"⚠️ Error during repo setup: {e}")
        return

    # Αντίγραφο για να μην πειράξεις το original df
    df_hub = df.copy()

    # --- Type fixes ---
    if "answer_index" in df_hub.columns:
        df_hub["answer_index"] = pd.to_numeric(
            df_hub["answer_index"], errors="coerce"
        ).astype("Int64")

    if "points" in df_hub.columns:
        df_hub["points"] = pd.to_numeric(df_hub["points"], errors="coerce")

    for col in ["id", "subject", "format", "reference", "question", "input",
                "answer_text", "image_description", "image_transcription",
                "year", "school_type"]:
        if col in df_hub.columns:
            df_hub[col] = df_hub[col].fillna("").astype(str)

    # Λίστες
    for col in ["choices", "images"]:
        if col in df_hub.columns:
            df_hub[col] = df_hub[col].apply(
                lambda x: x if isinstance(x, list) else ([] if pd.isna(x) else [x])
            )

    # --- Create HF dataset ---
    try:
        dataset = Dataset.from_pandas(df_hub, preserve_index=False)
    except Exception as e:
        print(f"❌ Failed to convert DataFrame to Dataset: {e}")
        return

    # --- Optional image casting ---
    if with_images:
        if "images" in dataset.column_names:
            print("🖼️ Casting 'images' column to Sequence(Image())...")
            try:
                dataset = dataset.cast_column("images", Sequence(Image()))
            except Exception as e:
                print(f"⚠️ Failed to cast 'images' as images: {e}")
                print("ℹ️ Continuing upload without image casting.")
        else:
            print("⚠️ Column 'images' not found, skipping image casting.")

    # --- Push to hub ---
    try:
        dataset.push_to_hub(
            repo_id,
            token=token,
            private=is_private,
            split="train"
        )
        print("✅ Successfully pushed to Hub (split='train').")
    except Exception as e:
        print(f"❌ Failed to push to Hub: {e}")"